In [1]:
##Logistics Delay Risk Analysis

# Objective
# Analyze shipment data to identify delay patterns, evaluate vendor performance, and detect high-risk shipments.

# Business Context
# Delays in supply chain lead to:

# Revenue loss
# Customer dissatisfaction
# Operational inefficiencies
# Dataset Overview
# Source: SCMS Delivery History Dataset
# Records: ~10K shipments
# Features: 30+ variables (vendor, shipment mode, pricing, etc.)
# Key Goals
# Measure % delayed shipments
# Identify high-risk vendors
# Analyze shipment patterns
# Support decision-making


In [2]:
import pandas as pd

RAW_DATA_PATH = ("../data/raw/SCMS_Delivery_History_Dataset.csv")

df = pd.read_csv(
    RAW_DATA_PATH,
    encoding='UTF-8-SIG'
)

print(df.shape)
df.head()

(10324, 33)


,ID,Project Code,PQ #,PO / SO #,ASN/DN #,Country,Managed By,Fulfill Via,Vendor INCO Term,Shipment Mode,...,Unit of Measure (Per Pack),Line Item Quantity,Line Item Value,Pack Price,Unit Price,Manufacturing Site,First Line Designation,Weight (Kilograms),Freight Cost (USD),Line Item Insurance (USD)
0,1,100-CI-T01,Pre-PQ Process,SCMS-4,ASN-8,Côte d'Ivoire,PMO - US,Direct Drop,EXW,Air,...,30,19,551.0,29.00,0.97,Ranbaxy Fine Chemicals LTD,Yes,13,780.34,NaN
1,3,108-VN-T01,Pre-PQ Process,SCMS-13,ASN-85,Vietnam,PMO - US,Direct Drop,EXW,Air,...,240,1000,6200.0,6.20,0.03,"Aurobindo Unit III, India",Yes,358,4521.5,NaN
2,4,100-CI-T01,Pre-PQ Process,SCMS-20,ASN-14,Côte d'Ivoire,PMO - US,Direct Drop,FCA,Air,...,100,500,40000.0,80.00,0.80,ABBVIE GmbH & Co.KG Wiesbaden,Yes,171,1653.78,NaN
3,15,108-VN-T01,Pre-PQ Process,SCMS-78,ASN-50,Vietnam,PMO - US,Direct Drop,EXW,Air,...,60,31920,127360.8,3.99,0.07,"Ranbaxy, Paonta Shahib, India",Yes,1855,16007.06,NaN
4,16,108-VN-T01,Pre-PQ Process,SCMS-81,ASN-55,Vietnam,PMO - US,Direct Drop,EXW,Air,...,60,38000,121600.0,3.20,0.05,"Aurobindo Unit III, India",Yes,7590,45450.08,NaN


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10324 entries, 0 to 10323
Data columns (total 33 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   ID                            10324 non-null  int64  
 1   Project Code                  10324 non-null  object 
 2   PQ #                          10324 non-null  object 
 3   PO / SO #                     10324 non-null  object 
 4   ASN/DN #                      10324 non-null  object 
 5   Country                       10324 non-null  object 
 6   Managed By                    10324 non-null  object 
 7   Fulfill Via                   10324 non-null  object 
 8   Vendor INCO Term              10324 non-null  object 
 9   Shipment Mode                 9964 non-null   object 
 10  PQ First Sent to Client Date  10324 non-null  object 
 11  PO Sent to Vendor Date        10324 non-null  object 
 12  Scheduled Delivery Date       10324 non-null  object 
 13  D

In [4]:
# date columns needed to calculate delay
date_cols = [
    "Scheduled Delivery Date",
    "Delivered to Client Date"
]

for col in date_cols:
    df[col] = pd.to_datetime(
        df[col],
        errors='coerce'
    )

C:\Users\ankit\AppData\Local\Temp\ipykernel_23364\904429006.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(
C:\Users\ankit\AppData\Local\Temp\ipykernel_23364\904429006.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(


In [5]:
df[date_cols].head()
df[date_cols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10324 entries, 0 to 10323
Data columns (total 2 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Scheduled Delivery Date   10324 non-null  datetime64[ns]
 1   Delivered to Client Date  10324 non-null  datetime64[ns]
dtypes: datetime64[ns](2)
memory usage: 161.4 KB


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10324 entries, 0 to 10323
Data columns (total 33 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   ID                            10324 non-null  int64         
 1   Project Code                  10324 non-null  object        
 2   PQ #                          10324 non-null  object        
 3   PO / SO #                     10324 non-null  object        
 4   ASN/DN #                      10324 non-null  object        
 5   Country                       10324 non-null  object        
 6   Managed By                    10324 non-null  object        
 7   Fulfill Via                   10324 non-null  object        
 8   Vendor INCO Term              10324 non-null  object        
 9   Shipment Mode                 9964 non-null   object        
 10  PQ First Sent to Client Date  10324 non-null  object        
 11  PO Sent to Vendor Date      

##Data Cleaning & Feature Engineering

In [7]:
# Drop columns that are not needed

drop_unneeded = [
    "ï»¿ID", "ID", "Project Code", "PQ #", "PO / SO #", "ASN/DN #",
    "PQ First Sent to Client Date", "PO Sent to Vendor Date",
]
df = df.drop(columns=drop_unneeded, errors="ignore")
print(df.shape)

(10324, 26)


In [8]:
# Convert financial and weight columns to numeric

df["Weight (Kilograms)"] = pd.to_numeric(df["Weight (Kilograms)"], errors='coerce')
df["Freight Cost (USD)"] = pd.to_numeric(df["Freight Cost (USD)"], errors='coerce')

In [9]:
df = df.dropna(subset=[
    "Scheduled Delivery Date",
    "Delivered to Client Date"
])

In [10]:
# Handle missing values in key numeric columns

df["Weight (Kilograms)"] = df["Weight (Kilograms)"].fillna(df["Weight (Kilograms)"].median())

df["Freight Cost (USD)"] = df["Freight Cost (USD)"].fillna(0)

In [11]:
# Delay = Actual Delivery Date - Scheduled Delivery Date

df["delay_days"] = (
    df["Delivered to Client Date"] - df["Scheduled Delivery Date"]
).dt.days

In [12]:
df["is_delayed"] = df["delay_days"] > 0

In [13]:
# Cost per kg-  logistics efficiency metric
df["cost_per_kg"] = df["Freight Cost (USD)"] / df["Weight (Kilograms)"]

# Value per item-  pricing efficiency
df["value_per_item"] = df["Line Item Value"] / df["Line Item Quantity"]

In [14]:
df["Shipment Mode"] = df["Shipment Mode"].fillna("Unknown")

In [15]:
df["Line Item Insurance (USD)"] = df["Line Item Insurance (USD)"].fillna(0)

In [16]:
df.isnull().sum()

Country                          0
Managed By                       0
Fulfill Via                      0
Vendor INCO Term                 0
Shipment Mode                    0
Scheduled Delivery Date          0
Delivered to Client Date         0
Delivery Recorded Date           0
Product Group                    0
Sub Classification               0
Vendor                           0
Item Description                 0
Molecule/Test Type               0
Brand                            0
Dosage                        1736
Dosage Form                      0
Unit of Measure (Per Pack)       0
Line Item Quantity               0
Line Item Value                  0
Pack Price                       0
Unit Price                       0
Manufacturing Site               0
First Line Designation           0
Weight (Kilograms)               0
Freight Cost (USD)               0
Line Item Insurance (USD)        0
delay_days                       0
is_delayed                       0
cost_per_kg         

In [ ]:
import os
from pathlib import Path

output_dir = Path("..") / "data" / "processed_data"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "scms_clean.pkl")
df.to_pickle(output_path)

print(f"Saved {output_path}", df.shape)

Saved ..\data\processed_data\scms_clean.pkl (10324, 30)
